# Project T: Classification of Real vs. Fake Faces using ResNet-18
**Group Name:** IowaGPT  
**Members:** Amit Boodhoo, Diego Liogon, Eva Singh, Kate Meyer

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import os

# Set device to GPU if available on your machine, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ResNet-18 expects images to be a specific size and normalized
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Point this to the folder where you unzipped the Kaggle dataset
data_dir = 'path_to_your_dataset_folder' 

# Load the dataset using PyTorch's ImageFolder
full_dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)
print(f"Classes found: {full_dataset.classes}")

In [ ]:
# Split the dataset (e.g., 70% train, 15% val, 15% test)
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

# DataLoaders handle feeding the images to the model in batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Load the pre-trained ResNet-18 model
model = models.resnet18(weights='DEFAULT')

# Freeze the early layers so we don't destroy the generic image features it already learned
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer to output exactly 2 classes (Real vs Fake) instead of 1000
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)

# Set up the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
# A simple training loop to update the final layer weights
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad() # Clear old gradients
        outputs = model(inputs) # Predict
        loss = criterion(outputs, labels) # Calculate error
        loss.backward() # Backpropagate
        optimizer.step() # Update weights
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
# Save the model weights to a file so it can be loaded in the Demo Notebook
torch.save(model.state_dict(), 'resnet18_fake_faces.pt')
print("Model saved successfully!")